Imports

In [2]:
from pathlib import Path
import random

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [3]:
IMG_DIR = Path("../FRUS")
MASK_DIR = Path("../Masks")

In [4]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Using: {DEVICE}", flush=True)

Using: cuda


In [5]:
SIZE = (256, 256)
BATCH_SIZE = 2
EPOCHS = 30
LR = 1e-3

print("DEVICE:", DEVICE)

DEVICE: cuda


In [16]:
class UltrasoundDataset(Dataset):
    def __init__(self, image_paths, mask_dir):
        self.image_paths = image_paths
        self.mask_dir = mask_dir

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_dir / img_path.name

        if not mask_path.exists():
            raise FileNotFoundError(mask_path)

        # Gray-scale images
        image = Image.open(img_path).convert("L")
        image = image.resize(
            SIZE, Image.Resampling.BILINEAR
        )
        image = np.array(image, dtype=np.float32) / 255.0

        # Binary masks
        mask = Image.open(mask_path).convert("L")
        mask = mask.resize(
            SIZE, Image.Resampling.NEAREST
        )
        mask = np.array(mask, dtype=np.uint8)
        mask = (mask > 127).astype(np.float32)

        # [H, W] -> [1, H, W]
        image = torch.from_numpy(image).unsqueeze(0)
        mask = torch.from_numpy(mask).unsqueeze(0)

        return image, mask

U-Net Architecture

In [6]:
def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, 3, padding=1),
        nn.ReLU(inplace=True),
    )

In [7]:
class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.pool = nn.MaxPool2d(2)

        self.enc1 = conv_block(1, 32)
        self.enc2 = conv_block(32, 64)
        self.enc3 = conv_block(64, 128)
        self.enc4 = conv_block(128, 256)

        self.bottleneck = conv_block(256, 512)

        self.up4 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec4 = conv_block(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec3 = conv_block(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec2 = conv_block(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.dec1 = conv_block(64, 32)

        self.out = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.dec4(
            torch.cat([self.up4(b), e4], dim=1)
        )
        d3 = self.dec3(
            torch.cat([self.up3(d4), e3], dim=1)
        )
        d2 = self.dec2(
            torch.cat([self.up2(d3), e2], dim=1)
        )
        d1 = self.dec1(
            torch.cat([self.up1(d2), e1], dim=1)
        )

        # return logits, not probabilities
        return self.out(d1)

Loss functions

In [19]:
def dice_loss(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)

    intersection = (probs * targets).sum(dim=(1, 2, 3))
    denominator = (
        probs.sum(dim=(1, 2, 3))
        + targets.sum(dim=(1, 2, 3))
    )

    dice = (2 * intersection + eps) / (denominator + eps)
    return 1 - dice.mean()

In [20]:
def combined_loss(logits, targets):
    bce = nn.functional.binary_cross_entropy_with_logits(
        logits, targets
    )
    return bce + dice_loss(logits, targets)


In [21]:
def dice_score(logits, targets, eps=1e-6):
    preds = torch.sigmoid(logits) > 0.5
    targets = targets > 0.5

    intersection = (preds & targets).sum(dim=(1, 2, 3))
    denominator = (
        preds.sum(dim=(1, 2, 3))
        + targets.sum(dim=(1, 2, 3))
    )

    dice = (2 * intersection + eps) / (denominator + eps)
    return dice.mean().item()

Load and prepare dataset

In [ ]:
image_paths = sorted(IMG_DIR.glob("*.png"))

assert len(image_paths) >= 2, (
    "You need at least two PNG images."
)

for p in image_paths:
    assert (MASK_DIR / p.name).exists(), (
        f"Didnt find a mask with name: {p.name}"
    )


random.shuffle(image_paths)

n_val = max(1, int(0.2 * len(image_paths)))
val_paths = image_paths[:n_val]
train_paths = image_paths[n_val:]

print(n_val)
print(val_paths)
print(train_paths)

train_ds = UltrasoundDataset(train_paths, MASK_DIR)
val_ds = UltrasoundDataset(val_paths, MASK_DIR)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False
)

print("Training:", len(train_ds))
print("Validation:", len(val_ds))

52
[WindowsPath('../FRUS/frame_0237.png'), WindowsPath('../FRUS/frame_0097.png'), WindowsPath('../FRUS/frame_0194.png'), WindowsPath('../FRUS/frame_0171.png'), WindowsPath('../FRUS/frame_0008.png'), WindowsPath('../FRUS/frame_0148.png'), WindowsPath('../FRUS/frame_0239.png'), WindowsPath('../FRUS/frame_0183.png'), WindowsPath('../FRUS/frame_0157.png'), WindowsPath('../FRUS/frame_0154.png'), WindowsPath('../FRUS/frame_0066.png'), WindowsPath('../FRUS/frame_0060.png'), WindowsPath('../FRUS/frame_0144.png'), WindowsPath('../FRUS/frame_0080.png'), WindowsPath('../FRUS/frame_0152.png'), WindowsPath('../FRUS/frame_0225.png'), WindowsPath('../FRUS/frame_0164.png'), WindowsPath('../FRUS/frame_0249.png'), WindowsPath('../FRUS/frame_0045.png'), WindowsPath('../FRUS/frame_0029.png'), WindowsPath('../FRUS/frame_0251.png'), WindowsPath('../FRUS/frame_0137.png'), WindowsPath('../FRUS/frame_0256.png'), WindowsPath('../FRUS/frame_0247.png'), WindowsPath('../FRUS/frame_0042.png'), WindowsPath('../FRUS/

Training

In [ ]:
model = UNet().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_val_dice = -1.0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0

    for images, masks in train_loader:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        optimizer.zero_grad()

        logits = model(images)
        loss = combined_loss(logits, masks)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    model.eval()
    val_dice = 0.0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(DEVICE)
            masks = masks.to(DEVICE)

            logits = model(images)
            val_dice += dice_score(logits, masks)

    train_loss /= len(train_loader)
    val_dice /= len(val_loader)

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"Loss: {train_loss:.4f} | "
        f"Val Dice: {val_dice:.4f}"
    )

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save(model.state_dict(), "unet_best051126.pth")

print("Best validation dice:", best_val_dice)
print("Model saved in unet_best051126.pth")

Epoch 01/30 | Loss: 1.3239 | Val Dice: 0.4423
Epoch 02/30 | Loss: 1.2534 | Val Dice: 0.4423
Epoch 03/30 | Loss: 1.2181 | Val Dice: 0.4423
Epoch 04/30 | Loss: 1.1952 | Val Dice: 0.4423
Epoch 05/30 | Loss: 1.2471 | Val Dice: 0.4231
Epoch 06/30 | Loss: 1.2695 | Val Dice: 0.4423
Epoch 07/30 | Loss: 1.1933 | Val Dice: 0.4423
Epoch 08/30 | Loss: 1.2175 | Val Dice: 0.2575
Epoch 09/30 | Loss: 1.1751 | Val Dice: 0.2160
Epoch 10/30 | Loss: 1.1716 | Val Dice: 0.1951
Epoch 11/30 | Loss: 1.1706 | Val Dice: 0.1982
Epoch 12/30 | Loss: 1.1740 | Val Dice: 0.1636
Epoch 13/30 | Loss: 1.1723 | Val Dice: 0.1993
Epoch 14/30 | Loss: 1.1636 | Val Dice: 0.2201
Epoch 15/30 | Loss: 1.1575 | Val Dice: 0.1462
Epoch 16/30 | Loss: 1.1629 | Val Dice: 0.1657
Epoch 17/30 | Loss: 1.1691 | Val Dice: 0.2001
Epoch 18/30 | Loss: 1.1605 | Val Dice: 0.1872
Epoch 19/30 | Loss: 1.1547 | Val Dice: 0.4423
Epoch 20/30 | Loss: 1.1623 | Val Dice: 0.2059
Epoch 21/30 | Loss: 1.2031 | Val Dice: 0.4423
Epoch 22/30 | Loss: 1.1520 | Val D

In [9]:
# Load the best model for evaluation
model = UNet().to(DEVICE)

# Load trained weights
model.load_state_dict(
    torch.load(
        "unet_best271026.pth",
        map_location=DEVICE,
        weights_only=True
    )
)

# Evaluation mode
model.eval()

print("Model loaded and set to evaluation mode")

Model loaded and set to evaluation mode
